## Import Libraries and Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.diagnostic import het_white
from statsmodels.stats.stattools import durbin_watson
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("Setup complete for IFRS and Audit Quality Analysis")

Libraries imported successfully!
Setup complete for IFRS and Audit Quality Analysis


# CELL 2: Load and Inspect Data

In [ ]:
from google.colab import files
uploaded = files.upload()
# For uploaded file:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())

print("\nDataset info:")
print(df.info())

# CELL 3: Data Cleaning and Preparation

In [ ]:

# Check for missing values
print("Missing values by column:")
print(df.isnull().sum())

# Basic data types and summary
print("\nSummary statistics:")
print(df.describe())

# Check unique values in key categorical variables
print("\nUnique values in key variables:")
print(f"Countries: {df['Country'].unique()}")
print(f"IFRS Years: {sorted(df['IFRS_Year'].unique())}")

# Check if optional columns exist
optional_columns = ['Adopter_Type', 'IFRS_Year_Num', 'IFRS_Adopted']
for col in optional_columns:
    if col in df.columns:
        if col == 'IFRS_Year_Num':
            print(f"IFRS Year Numbers: {sorted(df[col].unique())}")
        else:
            print(f"{col}: {df[col].unique()}")
    else:
        print(f"⚠ Column '{col}' not found in dataset")

# CELL 4: Create Analysis Variables

In [ ]:
# Create log of audit fees for regression analysis
df['ln_Audit_Fees'] = np.log(df['Audit_Fees'])

# Create IFRS_Year_Num if it doesn't exist
if 'IFRS_Year_Num' not in df.columns:
    print("Creating IFRS_Year_Num from IFRS_Year...")
    # Map IFRS_Year to numeric values
    ifrs_mapping = {
        'IFRS-2': -2,
        'IFRS-1': -1,
        'IFRS0': 0,
        'IFRS+1': 1,
        'IFRS+2': 2
    }
    df['IFRS_Year_Num'] = df['IFRS_Year'].map(ifrs_mapping)
    print("✓ IFRS_Year_Num created")

# Create IFRS_Adopted if it doesn't exist
if 'IFRS_Adopted' not in df.columns:
    print("Creating IFRS_Adopted variable...")
    df['IFRS_Adopted'] = (df['IFRS_Year_Num'] > 0).astype(int)
    print("✓ IFRS_Adopted created (1 for post-adoption periods)")

# Create period indicator variables
df['Pre_IFRS'] = (df['IFRS_Year_Num'] < 0).astype(int)
df['Post_IFRS'] = (df['IFRS_Year_Num'] > 0).astype(int)
df['Adoption_Year'] = (df['IFRS_Year_Num'] == 0).astype(int)

# Create specific period dummies for event study
df['IFRS_minus2'] = (df['IFRS_Year_Num'] == -2).astype(int)
df['IFRS_minus1'] = (df['IFRS_Year_Num'] == -1).astype(int)
df['IFRS_0'] = (df['IFRS_Year_Num'] == 0).astype(int)  # Reference period
df['IFRS_plus1'] = (df['IFRS_Year_Num'] == 1).astype(int)
df['IFRS_plus2'] = (df['IFRS_Year_Num'] == 2).astype(int)

# Winsorize continuous variables at 1st and 99th percentiles
def winsorize(series, lower=0.01, upper=0.99):
    return np.clip(series, series.quantile(lower), series.quantile(upper))

# Apply winsorization to key continuous variables
continuous_vars = ['Audit_Fees', 'Firm_Size', 'Return_on_Assets', 'Leverage', 'Audit_Lags']
for var in continuous_vars:
    df[f'{var}_winsorized'] = winsorize(df[var])

# Create log of winsorized audit fees
df['ln_Audit_Fees_winsorized'] = np.log(df['Audit_Fees_winsorized'])

print("Variables created successfully!")
print("\nNew variables summary:")
print(df[['ln_Audit_Fees', 'Pre_IFRS', 'Post_IFRS', 'IFRS_minus1', 'IFRS_plus1']].describe())


# CELL 5: Descriptive Statistics


In [ ]:
# Summary statistics by IFRS adoption status
print("=== DESCRIPTIVE STATISTICS ===")
print("\nSummary by IFRS Adoption Status:")
summary_stats = df.groupby('IFRS_Adopted').agg({
    'Audit_Fees': ['count', 'mean', 'median', 'std'],
    'Audit_Lags': ['mean', 'median', 'std'],
    'Auditor_Switch': ['mean', 'sum'],
    'Big4': ['mean'],
    'Firm_Size': ['mean', 'median'],
    'Profitability': ['mean'],
    'Return_on_Assets': ['mean'],
    'Leverage': ['mean']
}).round(3)

print(summary_stats)

# Summary by time period relative to IFRS adoption
print("\nSummary by Time Period (IFRS_Year_Num):")
period_stats = df.groupby('IFRS_Year_Num').agg({
    'Audit_Fees': ['count', 'mean', 'std'],
    'Auditor_Switch': ['mean'],
    'Big4': ['mean'],
    'ln_Audit_Fees': ['mean']
}).round(3)

print(period_stats)


# CELL 6: Visualization - Audit Fees Trends

In [ ]:
# Create visualizations
plt.style.use('seaborn-v0_8')
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Audit fees over time
df_plot = df.groupby('IFRS_Year_Num')['Audit_Fees'].mean().reset_index()
axes[0,0].plot(df_plot['IFRS_Year_Num'], df_plot['Audit_Fees'], marker='o', linewidth=2, markersize=8)
axes[0,0].axvline(x=0, color='red', linestyle='--', alpha=0.7, label='IFRS Adoption')
axes[0,0].set_xlabel('Years Relative to IFRS Adoption')
axes[0,0].set_ylabel('Average Audit Fees')
axes[0,0].set_title('Audit Fees Around IFRS Adoption')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Log audit fees over time
df_plot_log = df.groupby('IFRS_Year_Num')['ln_Audit_Fees'].mean().reset_index()
axes[0,1].plot(df_plot_log['IFRS_Year_Num'], df_plot_log['ln_Audit_Fees'], marker='s', linewidth=2, markersize=8, color='green')
axes[0,1].axvline(x=0, color='red', linestyle='--', alpha=0.7, label='IFRS Adoption')
axes[0,1].set_xlabel('Years Relative to IFRS Adoption')
axes[0,1].set_ylabel('Average Log(Audit Fees)')
axes[0,1].set_title('Log Audit Fees Around IFRS Adoption')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Auditor switching rate
switch_rate = df.groupby('IFRS_Year_Num')['Auditor_Switch'].mean().reset_index()
axes[1,0].bar(switch_rate['IFRS_Year_Num'], switch_rate['Auditor_Switch'], alpha=0.7, color='orange')
axes[1,0].axvline(x=0, color='red', linestyle='--', alpha=0.7, label='IFRS Adoption')
axes[1,0].set_xlabel('Years Relative to IFRS Adoption')
axes[1,0].set_ylabel('Auditor Switch Rate')
axes[1,0].set_title('Auditor Switching Around IFRS Adoption')
axes[1,0].legend()

# Plot 4: Big4 usage rate
big4_rate = df.groupby('IFRS_Year_Num')['Big4'].mean().reset_index()
axes[1,1].bar(big4_rate['IFRS_Year_Num'], big4_rate['Big4'], alpha=0.7, color='purple')
axes[1,1].axvline(x=0, color='red', linestyle='--', alpha=0.7, label='IFRS Adoption')
axes[1,1].set_xlabel('Years Relative to IFRS Adoption')
axes[1,1].set_ylabel('Big4 Usage Rate')
axes[1,1].set_title('Big4 Auditor Usage Around IFRS Adoption')
axes[1,1].legend()

plt.tight_layout()
plt.show()


# CELL 7: Pre-Analysis Tests

In [ ]:
print("=== PRE-ANALYSIS TESTS ===")

# Test for balanced panel
firms_per_period = df.groupby('IFRS_Year_Num')['Firm_ID'].nunique()
print("Number of firms per period:")
print(firms_per_period)

total_firms = df['Firm_ID'].nunique()
total_periods = df['IFRS_Year_Num'].nunique()
print(f"\nTotal unique firms: {total_firms}")
print(f"Total periods: {total_periods}")
print(f"Expected observations for balanced panel: {total_firms * total_periods}")
print(f"Actual observations: {len(df)}")

if len(df) == total_firms * total_periods:
    print("✓ Dataset is balanced")
else:
    print("⚠ Dataset is unbalanced")

# Correlation matrix of key variables
corr_vars = ['ln_Audit_Fees', 'IFRS_Adopted', 'Firm_Size', 'Profitability',
             'Return_on_Assets', 'Leverage', 'Audit_Lags', 'Big4']
correlation_matrix = df[corr_vars].corr().round(3)
print("\nCorrelation Matrix:")
print(correlation_matrix)

# CELL 8: Main Regression Analysis - Audit Fees


In [ ]:
print("=== MAIN REGRESSION ANALYSIS ===")

# Model 1: Basic IFRS effect on log audit fees
formula1 = 'ln_Audit_Fees ~ IFRS_Adopted + Firm_Size + Profitability + Return_on_Assets + Leverage + Audit_Lags'
model1 = ols(formula1, data=df).fit()

print("MODEL 1: Basic IFRS Effect on Log Audit Fees")
print("=" * 50)
print(model1.summary())

# Calculate clustered standard errors (firm-level clustering)
# Note: For proper clustered SEs, you might want to use linearmodels package
# For now, we'll use robust standard errors
model1_robust = ols(formula1, data=df).fit(cov_type='HC1')
print("\nMODEL 1 with Robust Standard Errors:")
print("=" * 40)
print(model1_robust.summary())

# CELL 9: Event Study Analysis


In [ ]:

print("=== EVENT STUDY ANALYSIS ===")

# Model 2: Event study with period dummies (excluding IFRS_0 as reference)
formula2 = '''ln_Audit_Fees ~ IFRS_minus2 + IFRS_minus1 + IFRS_plus1 + IFRS_plus2 +
              Firm_Size + Profitability + Return_on_Assets + Leverage + Audit_Lags'''
model2 = ols(formula2, data=df).fit(cov_type='HC1')

print("MODEL 2: Event Study Analysis")
print("=" * 35)
print(model2.summary())

# Extract coefficients for plotting
event_coeffs = []
event_periods = [-2, -1, 0, 1, 2]  # 0 is reference (coefficient = 0)
event_se = []

# Get coefficients and standard errors
coeffs_dict = model2.params.to_dict()
se_dict = model2.bse.to_dict()

for period in event_periods:
    if period == 0:  # Reference period
        event_coeffs.append(0)
        event_se.append(0)
    else:
        var_name = f'IFRS_plus{period}' if period > 0 else f'IFRS_minus{abs(period)}'
        event_coeffs.append(coeffs_dict.get(var_name, 0))
        event_se.append(se_dict.get(var_name, 0))

# Plot event study results
plt.figure(figsize=(10, 6))
plt.errorbar(event_periods, event_coeffs, yerr=[1.96*se for se in event_se],
             marker='o', capsize=5, capthick=2, linewidth=2, markersize=8)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='IFRS Adoption')
plt.xlabel('Years Relative to IFRS Adoption')
plt.ylabel('Coefficient (Log Audit Fees)')
plt.title('Event Study: Effect of IFRS Adoption on Audit Fees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


# CELL 10: Auditor Choice Analysis

In [ ]:
print("=== AUDITOR CHOICE ANALYSIS ===")

# Check variation in Big4 variable
print("Big4 Auditor Distribution:")
print(df['Big4'].value_counts())
print(f"Big4 Usage Rate: {df['Big4'].mean():.3f}")

# Model 3: IFRS effect on Big4 choice (if there's variation)
if df['Big4'].var() > 0:  # Only run if there's variation
    # Use logistic regression for binary Big4 variable
    from statsmodels.discrete.discrete_model import Logit

    # Prepare data for logistic regression
    X_vars = ['IFRS_Adopted', 'Firm_Size', 'Profitability', 'Return_on_Assets', 'Leverage']
    X = df[X_vars].copy()
    X = sm.add_constant(X)  # Add intercept
    y = df['Big4']

    model3 = Logit(y, X).fit()
    print("MODEL 3: IFRS Effect on Big4 Auditor Choice (Logistic Regression)")
    print("=" * 60)
    print(model3.summary())

    # Calculate marginal effects
    marginal_effects = model3.get_margeff()
    print("\nMarginal Effects:")
    print(marginal_effects.summary())
else:
    print("⚠ No variation in Big4 variable - all firms use Big4 auditors")
    print("Cannot estimate Big4 choice model")

# Model 4: IFRS effect on auditor switching
formula4 = 'Auditor_Switch ~ IFRS_Adopted + Firm_Size + Profitability + Return_on_Assets + Leverage'
model4 = ols(formula4, data=df).fit(cov_type='HC1')

print("\nMODEL 4: IFRS Effect on Auditor Switching")
print("=" * 40)
print(model4.summary())

# CELL 11: Robustness Tests


In [ ]:
print("=== ROBUSTNESS TESTS ===")

# Test 1: Parallel trends test (pre-adoption periods)
pre_adoption_data = df[df['IFRS_Year_Num'] < 0].copy()
if len(pre_adoption_data) > 0:
    formula_pt = 'ln_Audit_Fees ~ IFRS_Year_Num + Firm_Size + Profitability + Return_on_Assets + Leverage'
    parallel_trends = ols(formula_pt, data=pre_adoption_data).fit()

    print("PARALLEL TRENDS TEST (Pre-adoption periods only)")
    print("=" * 50)
    print(f"IFRS_Year_Num coefficient: {parallel_trends.params['IFRS_Year_Num']:.4f}")
    print(f"P-value: {parallel_trends.pvalues['IFRS_Year_Num']:.4f}")

    if parallel_trends.pvalues['IFRS_Year_Num'] > 0.05:
        print("✓ Parallel trends assumption likely satisfied (p > 0.05)")
    else:
        print("⚠ Potential violation of parallel trends assumption (p < 0.05)")

# Test 2: Alternative specifications
print("\n" + "=" * 50)
print("ROBUSTNESS TEST: Alternative Specifications")

# Model with interaction terms
df['IFRS_Size_interaction'] = df['IFRS_Adopted'] * df['Firm_Size']
formula5 = '''ln_Audit_Fees ~ IFRS_Adopted + Firm_Size + IFRS_Size_interaction +
              Profitability + Return_on_Assets + Leverage + Audit_Lags'''
model5 = ols(formula5, data=df).fit(cov_type='HC1')

print("\nMODEL 5: IFRS Effect with Size Interaction")
print("=" * 40)
print(model5.summary())

# Test 3: Using winsorized variables
formula6 = '''ln_Audit_Fees_winsorized ~ IFRS_Adopted + Firm_Size_winsorized +
              Profitability + Return_on_Assets_winsorized + Leverage_winsorized + Audit_Lags_winsorized'''
model6 = ols(formula6, data=df).fit(cov_type='HC1')

print("\nMODEL 6: Using Winsorized Variables")
print("=" * 35)
print(model6.summary())

# CELL 12: Results Summary and Interpretation


In [ ]:
print("=== RESULTS SUMMARY ===")

# Create summary table of key results
results_summary = pd.DataFrame({
    'Model': ['Basic IFRS Effect', 'Event Study (IFRS+1)', 'Event Study (IFRS+2)',
              'Auditor Switching', 'With Interaction'],
    'Dependent_Variable': ['ln(Audit_Fees)', 'ln(Audit_Fees)', 'ln(Audit_Fees)',
                          'Auditor_Switch', 'ln(Audit_Fees)'],
    'IFRS_Coefficient': [
        model1_robust.params.get('IFRS_Adopted', np.nan),
        coeffs_dict.get('IFRS_plus1', np.nan),
        coeffs_dict.get('IFRS_plus2', np.nan),
        model4.params.get('IFRS_Adopted', np.nan),
        model5.params.get('IFRS_Adopted', np.nan)
    ],
    'Standard_Error': [
        model1_robust.bse.get('IFRS_Adopted', np.nan),
        se_dict.get('IFRS_plus1', np.nan),
        se_dict.get('IFRS_plus2', np.nan),
        model4.bse.get('IFRS_Adopted', np.nan),
        model5.bse.get('IFRS_Adopted', np.nan)
    ],
    'P_Value': [
        model1_robust.pvalues.get('IFRS_Adopted', np.nan),
        model2.pvalues.get('IFRS_plus1', np.nan),
        model2.pvalues.get('IFRS_plus2', np.nan),
        model4.pvalues.get('IFRS_Adopted', np.nan),
        model5.pvalues.get('IFRS_Adopted', np.nan)
    ]
})

# Add significance stars
def add_significance_stars(p_val):
    if pd.isna(p_val):
        return ''
    elif p_val < 0.01:
        return '***'
    elif p_val < 0.05:
        return '**'
    elif p_val < 0.10:
        return '*'
    else:
        return ''

results_summary['Significance'] = results_summary['P_Value'].apply(add_significance_stars)
results_summary = results_summary.round(4)

print("Key Results Summary:")
print("=" * 80)
print(results_summary.to_string(index=False))
print("\nSignificance levels: *** p<0.01, ** p<0.05, * p<0.10")

# CELL 13: Economic Significance Analysis


In [ ]:
print("\n=== ECONOMIC SIGNIFICANCE ANALYSIS ===")

# Calculate percentage changes in audit fees
ifrs_coeff = model1_robust.params.get('IFRS_Adopted', 0)
percentage_change = (np.exp(ifrs_coeff) - 1) * 100

print(f"IFRS Adoption Effect on Audit Fees:")
print(f"Log coefficient: {ifrs_coeff:.4f}")
print(f"Percentage change in audit fees: {percentage_change:.2f}%")

# Calculate dollar impact using mean audit fees
mean_audit_fees = df['Audit_Fees'].mean()
dollar_impact = mean_audit_fees * (np.exp(ifrs_coeff) - 1)

print(f"Average audit fees: ${mean_audit_fees:,.2f}")
print(f"Estimated dollar impact per firm: ${dollar_impact:,.2f}")

# Statistical significance interpretation
p_value = model1_robust.pvalues.get('IFRS_Adopted', 1)
print(f"Statistical significance (p-value): {p_value:.4f}")

if p_value < 0.05:
    print("✓ The effect is statistically significant at 5% level")
else:
    print("⚠ The effect is not statistically significant at 5% level")

# CELL 14: Export Results

In [ ]:
print("=== EXPORTING RESULTS ===")

# Create a comprehensive results dataset
results_data = df.groupby(['Firm_ID', 'IFRS_Year_Num']).agg({
    'Audit_Fees': 'first',
    'ln_Audit_Fees': 'first',
    'IFRS_Adopted': 'first',
    'Auditor_Switch': 'first',
    'Big4': 'first',
    'Firm_Size': 'first',
    'Country': 'first'
}).reset_index()

# Add predicted values from main model
df['predicted_ln_fees'] = model1_robust.fittedvalues
df['residuals'] = model1_robust.resid

# Export main results
results_export = df[['Firm_ID', 'Country', 'IFRS_Year', 'IFRS_Year_Num', 'IFRS_Adopted',
                    'Audit_Fees', 'ln_Audit_Fees', 'predicted_ln_fees', 'residuals',
                    'Auditor_Switch', 'Big4', 'Firm_Size', 'Profitability']].copy()

# Save results to CSV (will be downloaded)
results_export.to_csv('ifrs_analysis_results.csv', index=False)
results_summary.to_csv('ifrs_regression_summary.csv', index=False)

print("✓ Results exported to CSV files:")
print("  - ifrs_analysis_results.csv (detailed data with predictions)")
print("  - ifrs_regression_summary.csv (summary of key results)")

print("\n=== ANALYSIS COMPLETE ===")
print("Key Findings:")
print("1. Check the coefficient on IFRS_Adopted in Model 1 for the main effect")
print("2. Review the event study plot for timing of effects")
print("3. Examine auditor switching patterns around adoption")
print("4. Consider economic significance alongside statistical significance")
print("5. Review robustness tests for sensitivity of results")

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# Load data
df = pd.read_csv('IFRS_Early_Adopters.csv')

# Prepare data
df['IFRS_Adopted'] = np.where(df['IFRS_Year'] != 'IFRS-0', 1, 0)
y = df['Auditor_Switch']
X = df[['IFRS_Adopted', 'Firm_Size', 'Profitability', 'Return_on_Assets', 'Leverage']]
X = sm.add_constant(X)

# Fit logistic regression
logit_model = sm.Logit(y, X).fit()
print(logit_model.summary())

# Add predicted probabilities
df['Predicted_Prob'] = logit_model.predict(X)

# ====== 1. Violin Plot ======
plt.figure(figsize=(10,6))
sns.violinplot(x='IFRS_Adopted', y='Predicted_Prob', data=df, inner='quartile', palette='muted')
plt.title('Predicted Probability of Auditor Switching by IFRS Adoption')
plt.xlabel('IFRS Adopted')
plt.ylabel('Predicted Probability')
plt.xticks([0, 1], ['No', 'Yes'])
plt.grid(axis='y')
plt.show()

# ====== 2. Predicted Probabilities vs. Actual Outcomes ======
plt.figure(figsize=(10,6))
sns.scatterplot(x=df['Predicted_Prob'], y=y, hue=df['IFRS_Adopted'], palette='coolwarm', alpha=0.7)
plt.axhline(y=0.5, color='gray', linestyle='--')
plt.xlabel('Predicted Probability')
plt.ylabel('Actual Auditor Switch')
plt.title('Predicted Probability vs. Actual Outcome')
plt.grid(True)
plt.legend(title='IFRS Adopted', labels=['No', 'Yes'])
plt.show()

# ====== 3. ROC Curve ======
fpr, tpr, thresholds = roc_curve(y, df['Predicted_Prob'])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0,1], [0,1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Auditor Switching Prediction')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv('IFRS_Early_Adopters.csv')

# Clean and prepare the data
# IFRS Adoption: Convert 'IFRS_Year' to a binary dummy (1 if IFRS adoption, 0 otherwise)
df['IFRS_Adopted'] = np.where(df['IFRS_Year'] != 'IFRS-0', 1, 0)

# Dependent variable
y = df['Auditor_Switch']

# Independent variables
X = df[['IFRS_Adopted', 'Firm_Size', 'Profitability', 'Return_on_Assets', 'Leverage']]
X = sm.add_constant(X)

# Logistic Regression
logit_model = sm.Logit(y, X).fit()

# Print summary
print(logit_model.summary())

# Odds Ratios
odds_ratios = pd.DataFrame({
    'Variable': X.columns,
    'Odds Ratio': np.exp(logit_model.params),
    'P-Value': logit_model.pvalues
})
print("\nOdds Ratios:\n", odds_ratios)

# Plot Predicted Probabilities
df['Predicted_Prob'] = logit_model.predict(X)
sns.boxplot(x='IFRS_Adopted', y='Predicted_Prob', data=df)
plt.title('Predicted Probability of Auditor Switching by IFRS Adoption')
plt.xlabel('IFRS Adopted')
plt.ylabel('Predicted Probability')
plt.show()
